<a href="https://colab.research.google.com/github/bahmedx/733/blob/main/Hands_On_Project_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Setup and Data Loading**

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn modules
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay

# Load the Adult Census Income dataset (Predicting Income >50K)
# This dataset is excellent for both classification and ethical bias evaluation
print("Fetching dataset...")
data = fetch_openml(data_id=1590, as_frame=True, parser='auto')
df = data.frame

# Display basic info
display(df.head())
print(f"Dataset shape: {df.shape}")

**Data Preprocessing & Splitting**

In [ ]:
# Define features (X) and target (y)
# Target is 'class' (>50K or <=50K)
X = df.drop('class', axis=1)
y = df['class'].apply(lambda x: 1 if x == '>50K' else 0) # Encode target as binary

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

# Create preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Split the data (75/25 split, stratified due to class imbalance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

**Baseline Model Development**

In [ ]:
# Establish a baseline using DummyClassifier (predicts the majority class)
baseline_clf = DummyClassifier(strategy='prior')
baseline_clf.fit(X_train, y_train)

baseline_acc = baseline_clf.score(X_test, y_test)
print(f"Baseline Accuracy (predicting majority class): {baseline_acc:.4f}")

**Model 1 - Logistic Regression (with Cross-Validation)**

In [ ]:
# Build a pipeline for Logistic Regression
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Perform 5-fold cross-validation
cv_scores = cross_val_score(log_reg_pipeline, X_train, y_train, cv=5, scoring='accuracy')
print(f"LogReg CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Fit the model and evaluate on test set
log_reg_pipeline.fit(X_train, y_train)
y_pred_log = log_reg_pipeline.predict(X_test)

**Model 2 - Random Forest (with Hyperparameter Tuning)**

In [ ]:
# Build a pipeline for Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# Define parameter grid for GridSearchCV
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [10, 20, None]
}

# Execute GridSearchCV
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters for Random Forest: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)

**Evaluation Metrics and Visualization**

In [ ]:
# Function to evaluate and print metrics
def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"--- {name} Evaluation ---")
    print(classification_report(y_true, y_pred))
    print(f"ROC-AUC Score: {roc_auc_score(y_true, y_prob):.4f}\n")

# Get probabilities for ROC
y_prob_log = log_reg_pipeline.predict_proba(X_test)[:, 1]
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

evaluate_model("Logistic Regression", y_test, y_pred_log, y_prob_log)
evaluate_model("Random Forest (Tuned)", y_test, y_pred_rf, y_prob_rf)

# Plot Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_log), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title("LogReg Confusion Matrix")
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title("Random Forest Confusion Matrix")
plt.show()

# Plot ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(log_reg_pipeline, X_test, y_test, ax=ax, name="Logistic Regression")
RocCurveDisplay.from_estimator(best_rf, X_test, y_test, ax=ax, name="Random Forest")
plt.title("ROC Curve Comparison")
plt.show()

**Ethical Check - Feature Importance**

In [ ]:
# Extract feature importance from the tuned Random Forest
rf_model = best_rf.named_steps['classifier']
categorical_encoder = best_rf.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']

# Reconstruct feature names
cat_features_names = categorical_encoder.get_feature_names_out(categorical_features)
all_feature_names = np.concatenate([numeric_features, cat_features_names])

# Create a DataFrame of feature importances
importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Plot Top 15 Features
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(15))
plt.title('Top 15 Feature Importances (Check for sensitive demographic reliance)')
plt.show()